# ============================================================
# Assignment 2 — RAV4 Image-to-Video Semantic Retrieval
# Run this in Google Colab with a T4 GPU runtime
# Runtime → Change runtime type → T4 GPU
# ============================================================




In [1]:
# ── CELL 1: Verify GPU ──────────────────────────────────────
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — go to Runtime → Change runtime type → T4 GPU!")

GPU available: True
Device: Tesla T4


In [2]:
# ── CELL 2: Install dependencies ────────────────────────────
!pip install -q ultralytics yt-dlp datasets huggingface_hub pyarrow pandas Pillow tqdm
!apt-get install -q ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.1/182.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 99.8 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [5]:
# CELL 3 — FIXED (shows output)
import os, json

VIDEO_URL  = "https://www.youtube.com/watch?v=YcvECxtXoxQ"
VIDEO_FILE = "input_video.mp4"
FRAMES_DIR = "frames"

# Download with visible output
if not os.path.exists(VIDEO_FILE):
    !yt-dlp -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]" -o "input_video.mp4" "https://www.youtube.com/watch?v=YcvECxtXoxQ"
else:
    print(f"Already downloaded: {VIDEO_FILE}")

# Extract frames with visible output
os.makedirs(FRAMES_DIR, exist_ok=True)
if not os.listdir(FRAMES_DIR):
    !ffmpeg -i input_video.mp4 -vf "fps=1/2" -q:v 2 frames/frame_%06d.jpg
else:
    print("Frames already extracted")

# Build manifest
frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith(".jpg"))
manifest = []
for i, fname in enumerate(frames):
    manifest.append({
        "frame_index":   i,
        "filename":      fname,
        "filepath":      os.path.join(FRAMES_DIR, fname),
        "timestamp_sec": round(i * 2.0, 2),
    })
with open("frame_manifest.json", "w") as f:
    json.dump(manifest, f)
print(f"✅ {len(frames)} frames extracted")
print(f"Duration covered: {manifest[-1]['timestamp_sec']:.0f}s")


Already downloaded: input_video.mp4
Frames already extracted
✅ 503 frames extracted
Duration covered: 1004s


In [4]:
from ultralytics import YOLO
YOLO("yolov8m-seg.pt")
print("✅ yolov8m weights ready")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ yolov8m weights ready


In [6]:
from ultralytics import YOLO
YOLO("yolov8l-seg.pt")
print("✅ yolov8l ready")

✅ yolov8l ready


In [ ]:
from ultralytics import YOLO
import torch

print("Downloading model weights...")
YOLO("yolov8l-seg.pt")  # large model — best accuracy on T4
print(f"✅ yolov8l weights ready")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Also install CLIP for embedding-based retrieval
import subprocess
subprocess.run(["pip", "install", "-q", "git+https://github.com/openai/CLIP.git"], check=True)
print("✅ CLIP installed")

In [8]:
# ================================================================
# CELL 4a: Fine-tune YOLOv8l on carparts dataset
# ~30 min on T4 — hands off, start your report intro while waiting
# ================================================================
from ultralytics import YOLO
import torch

device = "0" if torch.cuda.is_available() else "cpu"
print(f"Training on: {torch.cuda.get_device_name(0)}")
print(f"VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB\n")

model = YOLO("yolov8l-seg.pt")

results = model.train(
    data="carparts-seg.yaml",
    epochs=30,
    imgsz=640,
    batch=8,              # fits T4 VRAM for large model
    device=device,
    project="runs/carparts",
    name="train",
    exist_ok=True,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,             # final LR = lr0 * lrf
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    patience=15,
    augment=True,         # mosaic, flips, HSV jitter etc.
    degrees=5.0,          # slight rotation augment — helps with varied viewpoints
    scale=0.3,            # scale augment — important for detecting small parts
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,            # mixup augmentation for better generalisation
    copy_paste=0.1,       # copy-paste augment for segmentation
    verbose=False,
    save=True,
    plots=True,
)

best_model = YOLO("runs/carparts/train/weights/best.pt")
print(f"\n✅ Fine-tuning complete!")
print(f"Classes ({len(best_model.names)}): {list(best_model.names.values())}")



Training on: Tesla T4
VRAM available: 15.6 GB

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=carparts-seg.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8l-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_m

FileNotFoundError: [Errno 2] No such file or directory: 'runs/carparts/train/weights/best.pt'

In [9]:
import glob
weights = glob.glob("/content/**/*.pt", recursive=True)
for w in weights:
    print(w)

/content/yolo26n.pt
/content/yolov8l-seg.pt
/content/yolov8m-seg.pt
/content/runs/segment/runs/carparts/train/weights/best.pt
/content/runs/segment/runs/carparts/train/weights/last.pt


In [10]:
best_model = YOLO("/content/runs/segment/runs/carparts/train/weights/best.pt")
print(f"✅ Fine-tuning complete!")
print(f"Classes ({len(best_model.names)}): {list(best_model.names.values())}")

✅ Fine-tuning complete!
Classes (23): ['back_bumper', 'back_door', 'back_glass', 'back_left_door', 'back_left_light', 'back_light', 'back_right_door', 'back_right_light', 'front_bumper', 'front_door', 'front_glass', 'front_left_door', 'front_left_light', 'front_light', 'front_right_door', 'front_right_light', 'hood', 'left_mirror', 'object', 'right_mirror', 'tailgate', 'trunk', 'wheel']


In [11]:
# ================================================================
# CELL 4b: Run detection + extract CLIP embeddings per detection
# For each frame:
#   1. Run YOLOv8l to get bounding boxes + class labels
#   2. Crop each detected region
#   3. Extract a CLIP embedding from the crop
#   4. Store everything in detections.parquet
# ================================================================
import json, os
import numpy as np
import pandas as pd
import torch
import clip
from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO

VIDEO_ID       = "YcvECxtXoxQ"
CONF_THRESHOLD = 0.30
OUTPUT_PARQUET = "detections.parquet"

device = "0" if torch.cuda.is_available() else "cpu"
torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load fine-tuned YOLOv8l
best_model = YOLO("/content/runs/segment/runs/carparts/train/weights/best.pt")
print(f"YOLOv8l classes: {list(best_model.names.values())}")

# Load CLIP for embedding extraction
clip_model, clip_preprocess = clip.load("ViT-B/32", device=torch_device)
clip_model.eval()
print(f"✅ CLIP ViT-B/32 loaded")

with open("frame_manifest.json") as f:
    manifest = json.load(f)
print(f"Processing {len(manifest)} frames...\n")

records = []

for entry in tqdm(manifest, desc="Detecting + Embedding", unit="frame"):
    frame_path = entry["filepath"]
    if not os.path.exists(frame_path):
        continue

    # ── YOLOv8 detection ──────────────────────────────────────
    results = best_model.predict(
        source=frame_path,
        conf=CONF_THRESHOLD,
        iou=0.45,
        device=device,
        augment=True,    # test-time augmentation for better recall
        verbose=False,
    )
    result = results[0]
    h, w   = result.orig_shape

    if result.boxes is None or len(result.boxes) == 0:
        continue

    # Load full frame for CLIP crops
    frame_img = Image.open(frame_path).convert("RGB")

    for box, conf, cls_id in zip(
        result.boxes.xyxy.cpu().numpy(),
        result.boxes.conf.cpu().numpy(),
        result.boxes.cls.cpu().numpy().astype(int),
    ):
        x1, y1, x2, y2 = [max(0, int(v)) for v in box]

        # ── CLIP embedding from detected crop ─────────────────
        crop = frame_img.crop((x1, y1, x2, y2))
        if crop.width < 10 or crop.height < 10:
            continue  # skip tiny crops

        clip_input = clip_preprocess(crop).unsqueeze(0).to(torch_device)
        with torch.no_grad():
            embedding = clip_model.encode_image(clip_input)
            embedding = embedding / embedding.norm(dim=-1, keepdim=True)  # L2 normalise
            embedding_np = embedding.cpu().numpy().flatten().tolist()

        records.append({
            "video_id":         VIDEO_ID,
            "frame_index":      entry["frame_index"],
            "timestamp_sec":    entry["timestamp_sec"],
            "class_label":      best_model.names[cls_id],
            "class_id":         int(cls_id),
            "confidence_score": float(round(conf, 4)),
            "x_min":            float(x1),
            "y_min":            float(y1),
            "x_max":            float(x2),
            "y_max":            float(y2),
            "detector_name":    "yolov8l-seg-carparts-finetuned",
            "frame_width":      w,
            "frame_height":     h,
            "clip_embedding":   embedding_np,   # 512-dim CLIP vector
        })

df = pd.DataFrame(records)
df.to_parquet(OUTPUT_PARQUET, index=False, engine="pyarrow")

print(f"\n✅ {len(df):,} detections saved → {OUTPUT_PARQUET}")
print(f"   Schema: {list(df.columns)}")
print(f"\nTop detected car parts:")
print(df["class_label"].value_counts().head(15).to_string())
print(f"\nFrames with detections: {df['frame_index'].nunique()} / {len(manifest)}")
print(f"Timestamp range: {df['timestamp_sec'].min():.1f}s – {df['timestamp_sec'].max():.1f}s")


YOLOv8l classes: ['back_bumper', 'back_door', 'back_glass', 'back_left_door', 'back_left_light', 'back_light', 'back_right_door', 'back_right_light', 'front_bumper', 'front_door', 'front_glass', 'front_left_door', 'front_left_light', 'front_light', 'front_right_door', 'front_right_light', 'hood', 'left_mirror', 'object', 'right_mirror', 'tailgate', 'trunk', 'wheel']


100%|███████████████████████████████████████| 338M/338M [00:06<00:00, 58.4MiB/s]


✅ CLIP ViT-B/32 loaded
Processing 503 frames...



Detecting + Embedding:   0%|          | 0/503 [00:00<?, ?frame/s]

WARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.


...

WARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.


Detecting + Embedding: 100%|█████████▉| 502/503 [00:57<00:00,  7.78frame/s]

WARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.


Detecting + Embedding: 100%|██████████| 503/503 [00:57<00:00,  8.78frame/s]



✅ 321 detections saved → detections.parquet
   Schema: ['video_id', 'frame_index', 'timestamp_sec', 'class_label', 'class_id', 'confidence_score', 'x_min', 'y_min', 'x_max', 'y_max', 'detector_name', 'frame_width', 'frame_height', 'clip_embedding']

Top detected car parts:
class_label
right_mirror        74
trunk               63
back_glass          43
wheel               38
front_glass         37
front_right_door    18
hood                 9
tailgate             8
front_bumper         6
back_bumper          5
back_left_door       4
back_left_light      4
back_right_door      4
left_mirror          3
front_left_door      2

Frames with detections: 220 / 503
Timestamp range: 0.0s – 1004.0s


In [15]:
# ── CELL 5: Two-stage retrieval ──────────────────────────────
import numpy as np
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
import torch, clip
from PIL import Image
from ultralytics import YOLO

# ── FIXED PATH ───────────────────────────────────────────────
BEST_MODEL_PATH = "/content/runs/segment/runs/carparts/train/weights/best.pt"

YOUTUBE_BASE       = "https://www.youtube.com/embed/YcvECxtXoxQ"
GAP_SEC            = 10.0
MIN_DETECTIONS     = 2
CONF_THRESHOLD     = 0.30
CLIP_SIM_THRESHOLD = 0.20

torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device       = "0" if torch.cuda.is_available() else "cpu"

video_df = pd.read_parquet("detections.parquet")
print(f"Video index: {len(video_df):,} detections")
print(f"Classes: {sorted(video_df['class_label'].unique())}")

print("\nBuilding CLIP embedding matrix...")
emb_matrix = np.stack(video_df["clip_embedding"].values).astype(np.float32)
print(f"Embedding matrix shape: {emb_matrix.shape}")

print("\nLoading models for query processing...")
retrieval_model = YOLO(BEST_MODEL_PATH)
clip_model, clip_preprocess = clip.load("ViT-B/32", device=torch_device)
clip_model.eval()
print("✅ Models loaded")

print("Loading query dataset...")
ds = load_dataset("aegean-ai/rav4-exterior-images", split="train")
print(f"Query images: {len(ds)}\n")

def merge_segments_weighted(subset_df, gap_sec):
    if subset_df.empty:
        return []
    subset_df = subset_df.sort_values("timestamp_sec")
    timestamps = subset_df["timestamp_sec"].tolist()
    confs      = subset_df["confidence_score"].tolist()
    sims       = subset_df["clip_similarity"].tolist()
    segments = []
    seg_start, seg_end = timestamps[0], timestamps[0]
    seg_confs, seg_sims = [confs[0]], [sims[0]]
    for t, c, s in zip(timestamps[1:], confs[1:], sims[1:]):
        if t - seg_end <= gap_sec:
            seg_end = t
            seg_confs.append(c)
            seg_sims.append(s)
        else:
            segments.append((seg_start, seg_end, seg_confs, seg_sims))
            seg_start, seg_end = t, t
            seg_confs, seg_sims = [c], [s]
    segments.append((seg_start, seg_end, seg_confs, seg_sims))
    return segments

rows = []

for i, sample in enumerate(tqdm(ds, desc="Retrieving", unit="query")):
    image  = sample["image"].convert("RGB")
    img_np = np.array(image)

    res = retrieval_model.predict(source=img_np, conf=CONF_THRESHOLD, device=device, augment=True, verbose=False)
    r   = res[0]
    if r.boxes is None or len(r.boxes) == 0:
        continue

    best_detections = {}
    for box, conf, cls_id in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int)):
        cls = retrieval_model.names[cls_id]
        if cls not in best_detections or conf > best_detections[cls]["conf"]:
            x1, y1, x2, y2 = [max(0, int(v)) for v in box]
            best_detections[cls] = {"conf": float(conf), "crop": image.crop((x1, y1, x2, y2))}

    for cls_label, det in best_detections.items():
        crop = det["crop"]
        if crop.width < 10 or crop.height < 10:
            continue
        clip_input = clip_preprocess(crop).unsqueeze(0).to(torch_device)
        with torch.no_grad():
            q_emb = clip_model.encode_image(clip_input)
            q_emb = q_emb / q_emb.norm(dim=-1, keepdim=True)
            q_emb_np = q_emb.cpu().numpy().flatten().astype(np.float32)

        class_mask    = video_df["class_label"] == cls_label
        if class_mask.sum() == 0:
            continue
        class_indices = np.where(class_mask)[0]
        similarities  = emb_matrix[class_indices] @ q_emb_np
        class_subset  = video_df[class_mask].copy()
        class_subset["clip_similarity"] = similarities
        class_subset  = class_subset[class_subset["clip_similarity"] >= CLIP_SIM_THRESHOLD]
        if class_subset.empty:
            continue

        for seg_start, seg_end, seg_confs, seg_sims in merge_segments_weighted(class_subset, GAP_SEC):
            n = len(seg_confs)
            if n < MIN_DETECTIONS:
                continue
            rows.append({
                "query_index":                     i,
                "query_timestamp":                 sample.get("timestamp", str(i)),
                "query_timestamp_sec":             sample.get("timestamp_sec", i),
                "query_detected_class":            cls_label,
                "query_confidence":                det["conf"],
                "start_timestamp":                 seg_start,
                "end_timestamp":                   seg_end,
                "class_label":                     cls_label,
                "number_of_supporting_detections": n,
                "mean_clip_similarity":            float(round(np.mean(seg_sims), 4)),
                "max_clip_similarity":             float(round(np.max(seg_sims), 4)),
                "mean_detection_confidence":       float(round(np.mean(seg_confs), 4)),
                "youtube_verify_url":              f"{YOUTUBE_BASE}?start={int(seg_start)}&end={int(seg_end)}",
                "video_id":                        "YcvECxtXoxQ",
            })

results_df = pd.DataFrame(rows)
if not results_df.empty:
    results_df = results_df.sort_values(["query_index", "mean_clip_similarity"], ascending=[True, False]).reset_index(drop=True)

results_df.to_parquet("retrieval_results.parquet", index=False)
print(f"\n✅ {len(results_df):,} retrieval results saved")
print("\nTop retrieved classes:")
print(results_df["class_label"].value_counts().head(10).to_string())
cols = ["query_timestamp", "class_label", "start_timestamp", "end_timestamp", "number_of_supporting_detections", "mean_clip_similarity"]
print("\nSample results:")
print(results_df[cols].head(10).to_string(index=False))

Video index: 321 detections
Classes: ['back_bumper', 'back_glass', 'back_left_door', 'back_left_light', 'back_light', 'back_right_door', 'front_bumper', 'front_glass', 'front_left_door', 'front_right_door', 'front_right_light', 'hood', 'left_mirror', 'right_mirror', 'tailgate', 'trunk', 'wheel']

Building CLIP embedding matrix...
Embedding matrix shape: (321, 512)

Loading models for query processing...
✅ Models loaded
Loading query dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/505 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/69.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/65 [00:00<?, ? examples/s]

...

WARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.


Retrieving:  98%|█████████▊| 64/65 [00:13<00:00,  7.86query/s]

WARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.


Retrieving: 100%|██████████| 65/65 [00:13<00:00,  4.85query/s]


✅ 946 retrieval results saved

Top retrieved classes:
class_label
right_mirror        312
wheel               198
front_glass         144
trunk                90
front_right_door     64
hood                 30
front_bumper         26
front_left_door      24
back_glass           21
back_left_door       15

Sample results:
query_timestamp     class_label  start_timestamp  end_timestamp  number_of_supporting_detections  mean_clip_similarity
          00:00  back_left_door              0.0            8.0                                3                0.8905
          00:00            hood              4.0            6.0                                2                0.8822
          00:00 front_left_door              6.0            8.0                                2                0.8710
          00:00     front_glass              0.0            6.0                                4                0.8708
          00:00     front_glass            138.0          142.0                  

In [16]:
# ================================================================
# CELL 6: Spot-check results visually
# ================================================================
print("── Top 5 results to verify on YouTube ─────────────────────")
top5 = results_df.nlargest(5, "mean_clip_similarity")
for _, row in top5.iterrows():
    print(f"\n  Class:      {row['class_label']}")
    print(f"  Query:      {row['query_timestamp']}")
    print(f"  Segment:    {row['start_timestamp']}s → {row['end_timestamp']}s")
    print(f"  CLIP sim:   {row['mean_clip_similarity']:.3f}")
    print(f"  Detections: {row['number_of_supporting_detections']}")
    print(f"  URL:        {row['youtube_verify_url']}")

── Top 5 results to verify on YouTube ─────────────────────

  Class:      hood
  Query:      03:25
  Segment:    4.0s → 6.0s
  CLIP sim:   0.921
  Detections: 2
  URL:        https://www.youtube.com/embed/YcvECxtXoxQ?start=4&end=6

  Class:      front_bumper
  Query:      04:20
  Segment:    4.0s → 6.0s
  CLIP sim:   0.915
  Detections: 2
  URL:        https://www.youtube.com/embed/YcvECxtXoxQ?start=4&end=6

  Class:      front_glass
  Query:      04:25
  Segment:    138.0s → 142.0s
  CLIP sim:   0.912
  Detections: 2
  URL:        https://www.youtube.com/embed/YcvECxtXoxQ?start=138&end=142

  Class:      left_mirror
  Query:      04:05
  Segment:    4.0s → 6.0s
  CLIP sim:   0.912
  Detections: 2
  URL:        https://www.youtube.com/embed/YcvECxtXoxQ?start=4&end=6

  Class:      trunk
  Query:      02:40
  Segment:    998.0s → 1004.0s
  CLIP sim:   0.910
  Detections: 2
  URL:        https://www.youtube.com/embed/YcvECxtXoxQ?start=998&end=1004


In [18]:
# ================================================================
# CELL 7: Upload detections.parquet to Hugging Face
# ================================================================
from huggingface_hub import HfApi, login
import os

HF_REPO = "naenile40/rav4-detections"  # ← CHANGE THIS

login()  # enter your HF token when prompted

api = HfApi()
api.create_repo(repo_id=HF_REPO, repo_type="dataset", exist_ok=True, private=False)

# Upload detection index (required deliverable)
api.upload_file(
    path_or_fileobj="detections.parquet",
    path_in_repo="data/detections.parquet",
    repo_id=HF_REPO,
    repo_type="dataset",
)

# Upload retrieval results too (good to have on record)
api.upload_file(
    path_or_fileobj="retrieval_results.parquet",
    path_in_repo="data/retrieval_results.parquet",
    repo_id=HF_REPO,
    repo_type="dataset",
)

# Upload a dataset card (README)
readme = f"""---
license: mit
task_categories:
  - object-detection
tags:
  - car-parts
  - video-retrieval
  - yolov8
  - clip
---

# RAV4 Exterior Video Detection Index

YOLOv8l-seg fine-tuned on the Ultralytics carparts-seg dataset, with CLIP ViT-B/32 embeddings
extracted per detection. Built for AI Spring 2026 Assignment 2.

## Pipeline
1. YOLOv8l-seg fine-tuned on `carparts-seg.yaml` (30 epochs, AdamW)
2. Frames extracted at 0.5fps from [RAV4 video](https://www.youtube.com/watch?v=YcvECxtXoxQ)
3. Per-detection CLIP ViT-B/32 embeddings for visual similarity retrieval
4. Two-stage retrieval: class label match → CLIP cosine similarity re-ranking

## detections.parquet Schema

| Field              | Type    | Description                              |
|--------------------|---------|------------------------------------------|
| video_id           | string  | YouTube video ID                         |
| frame_index        | int     | Frame number (0-based)                   |
| timestamp_sec      | float   | Time offset in seconds                   |
| class_label        | string  | Detected car part class                  |
| class_id           | int     | Numeric class id                         |
| confidence_score   | float   | YOLOv8 detection confidence              |
| x_min              | float   | Bounding box left (px)                   |
| y_min              | float   | Bounding box top (px)                    |
| x_max              | float   | Bounding box right (px)                  |
| y_max              | float   | Bounding box bottom (px)                 |
| detector_name      | string  | Model identifier                         |
| frame_width        | int     | Frame width in pixels                    |
| frame_height       | int     | Frame height in pixels                   |
| clip_embedding     | list    | 512-dim CLIP ViT-B/32 embedding (L2 norm)|
"""

with open("/tmp/README.md", "w") as f:
    f.write(readme)
api.upload_file(
    path_or_fileobj="/tmp/README.md",
    path_in_repo="README.md",
    repo_id=HF_REPO,
    repo_type="dataset",
)

print(f"\n✅ Uploaded → https://huggingface.co/datasets/{HF_REPO}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  detections.parquet          : 100%|##########|  408kB /  408kB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  retrieval_results.parquet   : 100%|##########| 31.6kB / 31.6kB            


✅ Uploaded → https://huggingface.co/datasets/naenile40/rav4-detections


In [19]:
# ================================================================
# CELL 8: Download outputs to local machine
# ================================================================
from google.colab import files
files.download("detections.parquet")
files.download("retrieval_results.parquet")
print("✅ Files downloaded — keep these for your report!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Files downloaded — keep these for your report!
